# 🗺️ RoadScan-MA — Cartographie GPS & Export KML
**Auteurs :** Mohamed Amine Belasri & Yahya Amajane  
**ENSAM Meknès — Projet académique IATD 2026**

## 📋 Description
Génération de cartes interactives Folium et export KML pour Google Maps.
Visualisation des dégradations routières géolocalisées sur Meknès.

---

In [ ]:
!pip install folium -q
import folium, random
from IPython.display import display

# ── Solution : utiliser CartoDB au lieu d'OpenStreetMap ──
carte = folium.Map(
    location=[33.8935, -5.5364],
    zoom_start=14,
    tiles="CartoDB positron"   # ← Change cette ligne
)

couleurs = {"critique": "red", "modere": "orange", "leger": "green"}
classes  = [("D40","critique"), ("D20","modere"), ("D10","leger"), ("D00","leger")]

for _ in range(40):
    lat = 33.8935 + random.uniform(-0.02, 0.02)
    lon = -5.5364 + random.uniform(-0.02, 0.02)
    cls, sev = random.choice(classes)
    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color=couleurs[sev],
        fill=True,
        fill_color=couleurs[sev],
        fill_opacity=0.8,
        tooltip=f"{cls} — {sev}"
    ).add_to(carte)

display(carte)

## ⚙️ 1. Installation & Imports
Installation de Folium et des dépendances cartographiques.

In [ ]:
# ── Installation ──────────────────────────────────────────
!pip install simplekml folium -q

import simplekml
import json
import os
from datetime import datetime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 📍 2. Données GPS des Dégradations
Chargement des coordonnées GPS des dégradations détectées.

In [ ]:
# Exemple de données — dans le vrai projet, ces données
# viennent directement de YOLOv8 + GPS
degradations = [
    {
        "id": 1,
        "classe": "D40",
        "severite": "critique",
        "lat": 33.8920,
        "lon": -5.5340,
        "confiance": 0.92,
        "date": "2026-04-22",
        "description": "Nid-de-poule profond"
    },
    {
        "id": 2,
        "classe": "D20",
        "severite": "modere",
        "lat": 33.8950,
        "lon": -5.5380,
        "confiance": 0.85,
        "date": "2026-04-22",
        "description": "Fissure en grille"
    },
    {
        "id": 3,
        "classe": "D10",
        "severite": "leger",
        "lat": 33.8910,
        "lon": -5.5360,
        "confiance": 0.78,
        "date": "2026-04-22",
        "description": "Fissure transversale"
    },
    {
        "id": 4,
        "classe": "D00",
        "severite": "leger",
        "lat": 33.8970,
        "lon": -5.5350,
        "confiance": 0.81,
        "date": "2026-04-22",
        "description": "Fissure longitudinale"
    },
    {
        "id": 5,
        "classe": "D40",
        "severite": "critique",
        "lat": 33.8935,
        "lon": -5.5400,
        "confiance": 0.95,
        "date": "2026-04-22",
        "description": "Grand nid-de-poule"
    },
]

## 🗺️ 3. Carte Folium Interactive
Génération de la carte avec marqueurs colorés par sévérité.

In [ ]:
def creer_kml(degradations, nom_fichier="roadscan_meknes.kml"):
    """
    Convertit les dégradations détectées en fichier KML
    lisible par Google Maps et Google Earth
    """

    # Créer l'objet KML
    kml = simplekml.Kml()
    kml.document.name = "RoadScan-MA — Degradations Routieres"
    kml.document.description = f"Inspection du {datetime.now().strftime('%d/%m/%Y')}"

    # ── Définir les styles par sévérité ──────────────────
    styles = {
        "critique": simplekml.Style(),
        "modere":   simplekml.Style(),
        "leger":    simplekml.Style(),
    }

    # Rouge pour critique
    styles["critique"].iconstyle.color = simplekml.Color.red
    styles["critique"].iconstyle.scale = 1.5
    styles["critique"].labelstyle.color = simplekml.Color.red

    # Orange pour modéré
    styles["modere"].iconstyle.color = simplekml.Color.orange
    styles["modere"].iconstyle.scale = 1.2
    styles["modere"].labelstyle.color = simplekml.Color.orange

    # Vert pour léger
    styles["leger"].iconstyle.color = simplekml.Color.green
    styles["leger"].iconstyle.scale = 1.0
    styles["leger"].labelstyle.color = simplekml.Color.green

    # ── Créer un dossier par sévérité ────────────────────
    dossiers = {
        "critique": kml.newfolder(name="Critique (D40 - Nids-de-poule)"),
        "modere":   kml.newfolder(name="Modere (D20 - Fissures grille)"),
        "leger":    kml.newfolder(name="Leger (D00/D10 - Fissures)"),
    }

    # ── Ajouter chaque dégradation ────────────────────────
    for d in degradations:
        dossier = dossiers[d["severite"]]

        # Créer le point GPS
        point = dossier.newpoint(
            name=f"{d['classe']} #{d['id']}",
            coords=[(d["lon"], d["lat"])]  # ATTENTION : lon avant lat dans KML
        )

        # Description détaillée (visible dans Google Maps)
        point.description = f"""
        Type        : {d['classe']}
        Severite    : {d['severite'].upper()}
        Description : {d['description']}
        Confiance   : {d['confiance']*100:.0f}%
        Date        : {d['date']}
        GPS         : {d['lat']}, {d['lon']}
        """

        # Appliquer le style couleur
        point.style = styles[d["severite"]]

    # ── Sauvegarder le fichier ───────────────────────────
    kml.save(nom_fichier)
    print(f"Fichier KML cree : {nom_fichier}")
    print(f"Total degradations : {len(degradations)}")
    return nom_fichier

## 📁 4. Export KML pour Google Maps
Export des dégradations au format KML pour visualisation Google Maps.

In [ ]:
# Générer le fichier KML
fichier_kml = creer_kml(degradations, "roadscan_meknes.kml")

# Télécharger depuis Colab
from google.colab import files
files.download(fichier_kml)
print("Fichier KML telecharge !")

Fichier KML cree : roadscan_meknes.kml
Total degradations : 5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Fichier KML telecharge !


In [ ]:
def generer_lien_gmaps(lat, lon, label="Degradation"):
    """Génère un lien direct Google Maps pour une dégradation"""
    return f"https://www.google.com/maps/place/{lat},{lon}/@{lat},{lon},18z"

# Afficher les liens pour toutes les dégradations critiques
print("=== DEGRADATIONS CRITIQUES ===")
for d in degradations:
    if d["severite"] == "critique":
        lien = generer_lien_gmaps(d["lat"], d["lon"], d["classe"])
        print(f"{d['classe']} #{d['id']} → {lien}")

=== DEGRADATIONS CRITIQUES ===
D40 #1 → https://www.google.com/maps/place/33.892,-5.534/@33.892,-5.534,18z
D40 #5 → https://www.google.com/maps/place/33.8935,-5.54/@33.8935,-5.54,18z
